# Причинный вывод

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Причинный вывод».

## Что делает метод

Корреляция не равна причинности. Причинный вывод оценивает по данным наблюдений эффект вмешательства: «что изменится, если вмешаться», а не «что наблюдается вместе». Главная трудность — смешивающие факторы (конфаундеры): признаки, влияющие сразу и на вмешательство, и на исход, из-за чего наивное сравнение групп даёт смещённую оценку.

Это нужно, когда рандомизированный эксперимент невозможен или дорог: оценка эффекта программы, лечения, изменения регламента по уже собранным данным.

В этом блокноте мы воспроизведём типичную ситуацию со смешивающим фактором и покажем, как корректировка на конфаундер восстанавливает истинный эффект. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы заметили: в дни, когда продаётся больше мороженого, число солнечных ожогов тоже растёт. Можно ли заключить, что мороженое вызывает ожоги? Нет — у обоих явлений есть общая причина: жаркая погода. Это и есть **конфаундер** (смешивающий фактор) — скрытая переменная, которая влияет одновременно и на то, что мы изучаем («вмешательство»), и на исход, из-за чего наивное сравнение даёт ложный результат.

В идеале причинный вопрос «что произойдёт, если сделать X» решается рандомизированным экспериментом: случайно назначить участникам вмешательство или контроль, и тогда конфаундеры «уравновесятся» между группами. Но рандомизация часто невозможна: нельзя случайно назначить болезнь, политику или климатическое воздействие.

Причинный вывод позволяет оценить эффект вмешательства по уже собранным наблюдательным данным, явно учитывая конфаундеры. Блокнот показывает разницу между наивной оценкой (смещённой из-за конфаундера) и двумя корректными методами.

**Конфаундер** — переменная, влияющая и на вероятность вмешательства, и на исход. Без учёта конфаундера причинный эффект искажается.

**Оценка склонности (propensity score)** — вероятность получить вмешательство, рассчитанная по наблюдаемым характеристикам объекта. Объекты с одинаковой склонностью, но разным вмешательством — «квазирандомизированные пары».

## 1. Установка библиотек

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Конструируем данные так, что **истинный причинный эффект вмешательства известен нам заранее** — это +2.0. Это позволит в конце проверить, какой метод даёт правильный ответ.

Сценарий: исследуется эффект некоторой программы на исход. Скрытая ловушка — `тяжесть` исходного состояния: более тяжёлые участники чаще попадают в программу (и потому группы несопоставимы), а сама тяжесть ухудшает исход независимо от программы. Именно из-за этого конфаундера наивное сравнение групп даст смещённую оценку.

После запуска ячейки выведется доля получивших вмешательство и первые строки таблицы.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1200

# Смешивающий фактор: тяжесть исходного состояния.
severity = rng.normal(0, 1, n)

# Вмешательство чаще получают участники с большей тяжестью.
p_treat = 1 / (1 + np.exp(-1.2 * severity))
treatment = (rng.random(n) < p_treat).astype(int)

# Истинный причинный эффект вмешательства равен +2.0.
TRUE_EFFECT = 2.0
outcome = (5.0
           + TRUE_EFFECT * treatment    # причинный вклад вмешательства
           - 3.0 * severity             # тяжесть ухудшает исход
           + rng.normal(0, 1.0, n))

data = pd.DataFrame({"тяжесть": severity, "вмешательство": treatment,
                     "результат": outcome})
print(f"Истинный причинный эффект вмешательства: +{TRUE_EFFECT}")
print(f"Доля получивших вмешательство: {treatment.mean():.2f}")
data.head()

## 4. Применение метода

### Шаг 1. Наивное сравнение групп — ошибочный, но распространённый путь

Просто сравним средний исход тех, кто получил вмешательство, и тех, кто не получил. Это интуитивно, но неверно: группы несопоставимы, потому что тяжёлые участники и чаще получают вмешательство, и имеют худший исход.

Ожидаем, что результат будет заметно ниже истинного эффекта +2.0 — конфаундер «тянет» оценку вниз.

In [ ]:
naive = (data.loc[data["вмешательство"] == 1, "результат"].mean()
         - data.loc[data["вмешательство"] == 0, "результат"].mean())
print(f"Наивная оценка (разность средних): {naive:+.3f}")
print(f"Истинный эффект:                   {TRUE_EFFECT:+.3f}")
print("Наивная оценка смещена: группа вмешательства состоит "
      "из более тяжёлых участников.")

### Шаг 2. Корректировка через регрессию с учётом конфаундера

Включаем конфаундер `тяжесть` в регрессию вместе с `вмешательством`. Теперь модель оценивает: какова разница в исходе между получившими и не получившими вмешательство **при одинаковой тяжести**? Это и есть причинный эффект при условии, что все конфаундеры учтены.

Коэффициент при переменной `вмешательство` должен быть близок к +2.0.

In [ ]:
from sklearn.linear_model import LinearRegression

X = data[["вмешательство", "тяжесть"]].to_numpy()
reg = LinearRegression().fit(X, data["результат"])
effect_reg = reg.coef_[0]
print(f"Оценка с поправкой на конфаундер (регрессия): {effect_reg:+.3f}")

### Шаг 3. Взвешивание по обратной вероятности вмешательства

Альтернативный подход: вместо того чтобы включать конфаундер в модель исхода, мы «пересоздаём» рандомизированный эксперимент статистически.

Для каждого участника вычисляем оценку склонности — вероятность попасть в группу вмешательства, исходя из его тяжести. Затем каждое наблюдение взвешивается обратно пропорционально этой вероятности: редкие случаи (тяжёлый участник без вмешательства) получают большой вес, типичные — малый.

После взвешивания группы становятся сопоставимы по конфаундеру, как если бы вмешательство было назначено случайно. Взвешенная разность средних — это оценка причинного эффекта.

Этот метод называют **IPW** (Inverse Probability Weighting) или взвешиванием по обратной вероятности.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Оценка склонности: вероятность вмешательства по конфаундеру.
ps_model = LogisticRegression()
ps_model.fit(data[["тяжесть"]], data["вмешательство"])
propensity = ps_model.predict_proba(data[["тяжесть"]])[:, 1]

# Вес обратной вероятности выравнивает группы.
t = data["вмешательство"].to_numpy()
weights = np.where(t == 1, 1 / propensity, 1 / (1 - propensity))

y = data["результат"].to_numpy()
effect_ipw = (np.average(y[t == 1], weights=weights[t == 1])
              - np.average(y[t == 0], weights=weights[t == 0]))
print(f"Оценка через взвешивание по склонности: {effect_ipw:+.3f}")

### Шаг 4. Сравнение всех трёх оценок

Строим столбчатую диаграмму: три оценки эффекта и истинное значение +2.0. Ключевой «ага»-момент: наивная оценка далека от истины, тогда как оба корректных метода восстанавливают её, несмотря на то что данные не рандомизированы.

In [ ]:
import matplotlib.pyplot as plt

names = ["Наивная\n(разность средних)", "Регрессия\nс поправкой",
         "Взвешивание\nпо склонности"]
values = [naive, effect_reg, effect_ipw]
colors = [VIZ["series"][2], VIZ["series"][0], VIZ["series"][1]]

fig, ax = plt.subplots(figsize=(10, 5.6))
bars = ax.bar(names, values, color=colors, width=0.6)
ax.axhline(TRUE_EFFECT, color=VIZ["series"][3], linestyle="--",
           linewidth=2, label=f"Истинный эффект (+{TRUE_EFFECT})")
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.15,
            f"{v:+.2f}", ha="center", fontsize=12, weight="bold")
ax.set_title("Оценки причинного эффекта вмешательства")
ax.set_ylabel("Оценка эффекта")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Как читать этот график.**
- Каждый столбец — одна оценка причинного эффекта вмешательства.
- Пунктирная горизонтальная линия — истинный эффект +2.0, заложенный при генерации данных.
- Число над столбцом — конкретное значение оценки.
- Оранжевый столбец (наивная оценка) должен быть заметно ниже пунктира: смещение от конфаундера.
- Синий и зелёный столбцы (с поправкой) должны быть близки к пунктиру.
- Если оба корректных метода дают схожие результаты, это дополнительное свидетельство достоверности оценки.

## 5. Интерпретация результата

- **Наивная оценка** заметно занижена: группа вмешательства состоит из более тяжёлых участников, а тяжесть сама ухудшает исход. Простое сравнение средних смешивает причинный эффект с влиянием конфаундера — это иллюстрация тезиса «корреляция не есть причинность».
- **Регрессия с поправкой** и **взвешивание по склонности** дают оценки, близкие к истинному эффекту +2.0. Оба метода устраняют смещение, но по-разному: регрессия моделирует исход, взвешивание моделирует назначение вмешательства.
- Согласие двух разных методов повышает доверие к результату — это стандартная практика проверки в причинном выводе.

Помните: корректность опирается на ключевое допущение — все существенные конфаундеры измерены и учтены. Если важный смешивающий фактор не вошёл в данные, смещение останется, и никакой метод его не устранит. Причинный вывод по наблюдениям всегда строится на содержательных предположениях о структуре связей.

## 6. Поэкспериментируйте сами

Измените один параметр и повторно запустите ячейки разделов 3 и 4.

**Что менять и что наблюдать:**
- Усильте конфаундер: замените `1.2 * severity` в строке `p_treat` на `2.5 * severity`. Насколько сильнее смещается наивная оценка?
- Уберите конфаундер из регрессии: в ячейке шага 2 используйте `X = data[["вмешательство"]].to_numpy()`. Что произойдёт с оценкой?
- Измените истинный эффект: `TRUE_EFFECT = 0.5`. Остаётся ли наивная оценка смещённой? Верно ли теперь восстанавливают эффект корректные методы?
- Добавьте второй конфаундер: `severity2 = rng.normal(0, 1, n)` и учтите его в модели вероятности вмешательства и в регрессии.

In [ ]:
# --- Шаблон загрузки своих данных ---
# data = pd.read_csv("my_observations.csv")
# treatment_col = "вмешательство"          # столбец 0/1
# outcome_col = "результат"                # столбец исхода
# confounders = ["возраст", "тяжесть"]     # измеренные конфаундеры
#
# X = data[[treatment_col] + confounders].to_numpy()
# reg = LinearRegression().fit(X, data[outcome_col])
# print("Оценка эффекта с поправкой:", reg.coef_[0])

## Краткие выводы

- Корреляция не равна причинности: наивное сравнение групп смешивает эффект вмешательства с влиянием конфаундеров.
- Конфаундер — переменная, влияющая одновременно на вмешательство и на исход. Его необходимо явно учитывать.
- Два базовых метода корректировки — регрессия с включением конфаундеров и взвешивание по обратной вероятности — дают схожие результаты и восстанавливают истинный эффект.
- Корректность любого метода зависит от того, все ли важные конфаундеры измерены. Неизмеренный конфаундер нельзя учесть никаким статистическим методом — это фундаментальное ограничение.
- Согласие двух независимых методов — признак доверия к результату; расхождение требует объяснения.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. В демо наивная оценка занижена по сравнению с истинным эффектом +2.0. Не всегда смещение направлено в сторону занижения. Опишите механизм, при котором конфаундер мог бы завысить наивную оценку, и приведите пример из области науки или медицины.
2. Оба корректных метода (регрессия и IPW) дали оценки, близкие к истинному значению, но друг другу они не равны. В каких условиях это расхождение должно насторожить исследователя?
3. Вы хотите оценить причинный эффект нового режима обработки на выход продукта, но в данных не зафиксировали температуру реактора, которая влияет и на выбор режима, и на выход. Можно ли применить описанные методы, и что следует сделать в этой ситуации?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Если группа вмешательства исходно находится в более благоприятных условиях (например, более молодые и здоровые пациенты чаще получают экспериментальное лечение), наивное сравнение преувеличит эффект лечения: часть наблюдаемого улучшения объясняется исходным состоянием, а не вмешательством. Пример: добровольцы на клиническое испытание — как правило, более здоровые, чем средний пациент.
2. Расхождение должно насторожить, если оно велико относительно величины самого эффекта. Это может означать нарушение допущений одного из методов: наличие взаимодействия между конфаундером и вмешательством (нелинейность), экстремальные веса IPW (нарушение overlap), или неизмеренные конфаундеры. Согласие двух методов служит косвенным свидетельством правильности, но не доказательством.
3. Нет: неизмеренный конфаундер не может быть учтён никаким статистическим методом. Следует либо получить данные о температуре (измерить её ретроспективно или в будущем), либо использовать метод инструментальных переменных при наличии подходящего инструмента, либо честно признать ограниченность оценки и указать направление возможного смещения.
</details>
